In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [106]:
PJME_data = pd.read_csv("processed_data/PJME_phase2_preprocessed.csv",parse_dates=["Datetime"])

In [107]:
PJME_data = PJME_data.set_index("Datetime")

# Sort chronologically
PJME_data = PJME_data.sort_index()

print(PJME_data.shape)
print(PJME_data.head())

(145224, 15)
                     PJME_MW  Hour  Day  Week  Month  DayOfWeek  IsWeekend  \
Datetime                                                                     
2002-01-08 01:00:00  29445.0     1    8     2      1          1          0   
2002-01-08 02:00:00  28670.0     2    8     2      1          1          0   
2002-01-08 03:00:00  28375.0     3    8     2      1          1          0   
2002-01-08 04:00:00  28542.0     4    8     2      1          1          0   
2002-01-08 05:00:00  29261.0     5    8     2      1          1          0   

                       Lag_1   Lag_24   Lag_48  Lag_168  Rolling_Mean_24  \
Datetime                                                                   
2002-01-08 01:00:00  31187.0  26862.0  27100.0  30393.0     33452.583333   
2002-01-08 02:00:00  29445.0  25976.0  26097.0  29265.0     33560.208333   
2002-01-08 03:00:00  28670.0  25641.0  25793.0  28357.0     33672.458333   
2002-01-08 04:00:00  28375.0  25666.0  25657.0  27899.0     

In [6]:
# Make sure data is chronological
PJME_data = PJME_data.sort_index()

n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = PJME_data.iloc[:train_end].copy()
validation = PJME_data.iloc[train_end:val_end].copy()
test = PJME_data.iloc[val_end:].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

print("\nTrain period:")
print(train.index.min(), "to", train.index.max())

print("\nValidation period:")
print(validation.index.min(), "to", validation.index.max())

print("\nTest period:")
print(test.index.min(), "to", test.index.max())

Train: (101656, 15)
Validation: (21784, 15)
Test: (21784, 15)

Train period:
2002-01-08 01:00:00 to 2013-08-13 16:00:00

Validation period:
2013-08-13 17:00:00 to 2016-02-07 08:00:00

Test period:
2016-02-07 09:00:00 to 2018-08-03 00:00:00


In [108]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

target = "PJME_MW"

X_train = train[features].copy()
X_val = validation[features].copy()
X_test = test[features].copy()

y_train = train[target].copy()
y_val = validation[target].copy()
y_test = test[target].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (101656, 15)
X_val: (21784, 15)
X_test: (21784, 15)


In [109]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(
    y_train.values.reshape(-1, 1)
)

y_val_scaled = target_scaler.transform(
    y_val.values.reshape(-1, 1)
)

y_test_scaled = target_scaler.transform(
    y_test.values.reshape(-1, 1)
)

print("Scaling completed.")

Scaling completed.


In [110]:
def create_sequences(X, y, input_steps=48, output_steps=24):

    X_seq = []
    y_seq = []

    for i in range(input_steps, len(X) - output_steps + 1):

        X_seq.append(
            X[i-input_steps:i]
        )

        y_seq.append(
            y[i:i+output_steps].flatten()
        )

    return np.array(X_seq), np.array(y_seq)


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=48,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=48,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=48,
    output_steps=24
)


print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101585, 48, 15)
y_train: (101585, 24)
X_val: (21713, 48, 15)
y_val: (21713, 24)
X_test: (21713, 48, 15)
y_test: (21713, 24)


**Early Stopping**

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [11]:
rnn_es_model = Sequential([
    SimpleRNN(64,input_shape=(48, 15),return_sequences=False),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_es_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_es_model.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,840 (42.34 KB)

 Trainable params: 10,840 (42.34 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [13]:
history_es = rnn_es_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1311 - mae: 0.2599 - val_loss: 0.0929 - val_mae: 0.2221
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0769 - mae: 0.1999 - val_loss: 0.0836 - val_mae: 0.2098
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0707 - mae: 0.1895 - val_loss: 0.0807 - val_mae: 0.2046
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0679 - mae: 0.1847 - val_loss: 0.0783 - val_mae: 0.2012
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0657 - mae: 0.1813 - val_loss: 0.0780 - val_mae: 0.2013
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0644 - mae: 0.1790 - val_loss: 0.0782 - val_mae: 0.2001
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0632 - mae: 0.1771 - val_loss: 0.0789 - val_mae: 0.2039
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0621 - mae: 0.1753 - val_loss: 0.0744 - val_mae: 0.1945
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━━━━━

In [14]:
rnn_es_pred_scaled = rnn_es_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [15]:
rnn_es_pred = target_scaler.inverse_transform(
    rnn_es_pred_scaled.reshape(-1, 1)
).reshape(rnn_es_pred_scaled.shape)

rnn_es_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(rnn_es_actual.flatten(),rnn_es_pred.flatten())
rmse = np.sqrt(mean_squared_error(rnn_es_actual.flatten(),rnn_es_pred.flatten()))
mape = np.mean(np.abs((rnn_es_actual.flatten() - rnn_es_pred.flatten())/ rnn_es_actual.flatten())) * 100
r2 = r2_score(rnn_es_actual.flatten(),rnn_es_pred.flatten())
bias = np.mean(rnn_es_pred.flatten() - rnn_es_actual.flatten())

print("==== RNN + EARLY STOPPING ====")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

==== RNN + EARLY STOPPING ====
MAE : 1391.5821917466278
RMSE: 1947.6467640337682
MAPE: 4.402527263388842
R²  : 0.9086603906880356
Bias: 115.37338664570908


In [17]:
print("Epochs completed:", len(history_es.history["loss"]))

Epochs completed: 11


**Dropout**

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout

rnn_dropout_model = Sequential([SimpleRNN(64,input_shape=(48, 15),return_sequences=False),
    Dropout(0.2),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(24)
])

rnn_dropout_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_dropout_model.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_2 (SimpleRNN)        │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,840 (42.34 KB)

 Trainable params: 10,840 (42.34 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
history_dropout = rnn_dropout_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.2044 - mae: 0.3375 - val_loss: 0.0963 - val_mae: 0.2311
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1317 - mae: 0.2751 - val_loss: 0.0925 - val_mae: 0.2254
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1246 - mae: 0.2665 - val_loss: 0.0928 - val_mae: 0.2221
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1215 - mae: 0.2630 - val_loss: 0.0842 - val_mae: 0.2121
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1181 - mae: 0.2589 - val_loss: 0.0854 - val_mae: 0.2120
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1169 - mae: 0.2575 - val_loss: 0.0854 - val_mae: 0.2130
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1152 - mae: 0.2555 - val_loss: 0.0847 - val_mae: 0.2138
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1141 - mae: 0.2540 - val_loss: 0.0816 - val_mae: 0.2066
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━━━━━

In [20]:
dropout_pred_scaled = rnn_dropout_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [21]:
dropout_pred = target_scaler.inverse_transform(
    dropout_pred_scaled.reshape(-1, 1)
).reshape(dropout_pred_scaled.shape)

dropout_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [22]:
mae = mean_absolute_error(dropout_actual.flatten(),dropout_pred.flatten())
rmse = np.sqrt(mean_squared_error(dropout_actual.flatten(), dropout_pred.flatten())
mape = np.mean(np.abs((dropout_actual.flatten() - dropout_pred.flatten())/ dropout_actual.flatten()) * 100
r2 = r2_score(dropout_actual.flatten(),dropout_pred.flatten())
bias = np.mean(dropout_pred.flatten() - dropout_actual.flatten())

print("=== RNN + DROPOUT ===")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

=== RNN + DROPOUT ===
MAE : 1495.8575128838067
RMSE: 2030.1111137647854
MAPE: 4.816662229575421
R²  : 0.9007619139103995
Bias: 322.1394060022126


**Batch Normalization**

In [23]:
from tensorflow.keras.layers import SimpleRNN, Dense, BatchNormalization

rnn_bn_model = Sequential([SimpleRNN(64,input_shape=(48, 15) ),
    BatchNormalization(),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(24)
])

rnn_bn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_bn_model.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_3 (SimpleRNN)        │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,352 (44.34 KB)

 Trainable params: 11,096 (43.34 KB)

 Non-trainable params: 256 (1.00 KB)

In [24]:
history_bn = rnn_bn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.1785 - mae: 0.3083 - val_loss: 0.1024 - val_mae: 0.2388
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1007 - mae: 0.2379 - val_loss: 0.0880 - val_mae: 0.2181
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0924 - mae: 0.2268 - val_loss: 0.0831 - val_mae: 0.2078
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0881 - mae: 0.2207 - val_loss: 0.0836 - val_mae: 0.2065
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0861 - mae: 0.2181 - val_loss: 0.0791 - val_mae: 0.2024
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0835 - mae: 0.2143 - val_loss: 0.0768 - val_mae: 0.2002
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0837 - mae: 0.2150 - val_loss: 0.0776 - val_mae: 0.1993
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0811 - mae: 0.2113 - val_loss: 0.0778 - val_mae: 0.2007
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━━━━━

In [25]:
bn_pred_scaled = rnn_bn_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [26]:
rnn_bn_pred = target_scaler.inverse_transform(
    bn_pred_scaled.reshape(-1, 1)
).reshape(rnn_es_pred_scaled.shape)

rnn_bn_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(rnn_bn_actual.flatten(), rnn_bn_pred.flatten())
rmse = np.sqrt(mean_squared_error(rnn_bn_actual.flatten(),rnn_bn_pred.flatten()))
mape = np.mean(np.abs( (rnn_bn_actual.flatten() - rnn_bn_pred.flatten())/ rnn_bn_actual.flatten())) * 100
r2 = r2_score(rnn_bn_actual.flatten(),rnn_bn_pred.flatten())
bias = np.mean(rnn_bn_pred.flatten() - rnn_bn_actual.flatten())

print("==== RNN + Batch ====")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

==== RNN + Batch ====
MAE : 1448.650549633674
RMSE: 2017.3572154669084
MAPE: 4.609178909509545
R²  : 0.90200489685922
Bias: 303.88435695490483


**ADAM**

In [28]:
from tensorflow.keras.optimizers import Adam

rnn_adam = Sequential([
    SimpleRNN(64, input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_adam.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [29]:
history_adam = rnn_adam.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1273 - mae: 0.2549 - val_loss: 0.0945 - val_mae: 0.2213
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0781 - mae: 0.2009 - val_loss: 0.0909 - val_mae: 0.2183
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0722 - mae: 0.1914 - val_loss: 0.0842 - val_mae: 0.2084
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0690 - mae: 0.1861 - val_loss: 0.0829 - val_mae: 0.2061
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0670 - mae: 0.1830 - val_loss: 0.0795 - val_mae: 0.1997
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0656 - mae: 0.1806 - val_loss: 0.0775 - val_mae: 0.1969
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0643 - mae: 0.1785 - val_loss: 0.0773 - val_mae: 0.1988
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0635 - mae: 0.1774 - val_loss: 0.0814 - val_mae: 0.2019
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━━━━━

In [30]:
adam_pred_scaled = rnn_adam.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [31]:
adam_pred = target_scaler.inverse_transform(
    adam_pred_scaled.reshape(-1, 1)
).reshape(adam_pred_scaled.shape)

adam_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [32]:
adam_mae = mean_absolute_error(adam_actual.flatten(),adam_pred.flatten())
adam_rmse = np.sqrt( mean_squared_error( adam_actual.flatten(), adam_pred.flatten()))
adam_mape = np.mean(np.abs( (adam_actual.flatten() - adam_pred.flatten())/ adam_actual.flatten())) * 100
adam_r2 = r2_score( adam_actual.flatten(),adam_pred.flatten())
adam_bias = np.mean(adam_pred.flatten() -adam_actual.flatten())

print("========== RNN + ADAM ==========")
print("MAE :", adam_mae)
print("RMSE:", adam_rmse)
print("MAPE:", adam_mape)
print("R²  :", adam_r2)
print("Bias:", adam_bias)

========== RNN + ADAM ==========
MAE : 1413.3843418354943
RMSE: 1983.6639953866656
MAPE: 4.456305317783067
R²  : 0.9052509239111448
Bias: 73.88685066067251


**RMSPROP**

In [33]:
from tensorflow.keras.optimizers import RMSprop

rnn_rmsprop = Sequential([
    SimpleRNN(
        64,
        input_shape=(48, 15),
        return_sequences=False
    ),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_rmsprop.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

rnn_rmsprop.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_5 (SimpleRNN)        │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,840 (42.34 KB)

 Trainable params: 10,840 (42.34 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
history_rmsprop = rnn_rmsprop.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1267 - mae: 0.2588 - val_loss: 0.1133 - val_mae: 0.2500
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0815 - mae: 0.2061 - val_loss: 0.0992 - val_mae: 0.2306
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0740 - mae: 0.1940 - val_loss: 0.0909 - val_mae: 0.2181
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0701 - mae: 0.1878 - val_loss: 0.0974 - val_mae: 0.2228
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0677 - mae: 0.1839 - val_loss: 0.0874 - val_mae: 0.2121
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0658 - mae: 0.1808 - val_loss: 0.0940 - val_mae: 0.2208
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0645 - mae: 0.1785 - val_loss: 0.0804 - val_mae: 0.2037
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0633 - mae: 0.1768 - val_loss: 0.0797 - val_mae: 0.2028
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━━━━━

In [35]:
rmsprop_pred_scaled = rnn_rmsprop.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rmsprop_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Prediction shape: (21713, 24)


In [36]:
rmsprop_pred = target_scaler.inverse_transform(
    rmsprop_pred_scaled.reshape(-1, 1)
).reshape(rmsprop_pred_scaled.shape)

rmsprop_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [37]:
rmsprop_mae = mean_absolute_error(rmsprop_actual.flatten(), rmsprop_pred.flatten())
rmsprop_rmse = np.sqrt( mean_squared_error(rmsprop_actual.flatten(),rmsprop_pred.flatten()))
rmsprop_mape = np.mean(np.abs((rmsprop_actual.flatten() - rmsprop_pred.flatten())/ rmsprop_actual.flatten())) * 100
rmsprop_r2 = r2_score(rmsprop_actual.flatten(),rmsprop_pred.flatten())
rmsprop_bias = np.mean(rmsprop_pred.flatten() - rmsprop_actual.flatten())

print("==== RNN + RMSPROP ====")
print("MAE :", rmsprop_mae)
print("RMSE:", rmsprop_rmse)
print("MAPE:", rmsprop_mape)
print("R²  :", rmsprop_r2)
print("Bias:", rmsprop_bias)

==== RNN + RMSPROP ====
MAE : 1368.0419974047086
RMSE: 1924.233401197823
MAPE: 4.3164857429288785
R²  : 0.9108432435636404
Bias: 30.372446078949917


**SGD**

In [45]:
from tensorflow.keras.optimizers import SGD

rnn_sgd = Sequential([SimpleRNN(64,input_shape=(48, 15),return_sequences=False),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_sgd.compile(optimizer=SGD(learning_rate=0.001,momentum=0.9),
    loss="mse",
    metrics=["mae"]
)

rnn_sgd.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_10 (SimpleRNN)       │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,840 (42.34 KB)

 Trainable params: 10,840 (42.34 KB)

 Non-trainable params: 0 (0.00 B)

In [46]:
history_sgd = rnn_sgd.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.3819 - mae: 0.4688 - val_loss: 0.2067 - val_mae: 0.3559
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1638 - mae: 0.3118 - val_loss: 0.1540 - val_mae: 0.3004
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1314 - mae: 0.2749 - val_loss: 0.1356 - val_mae: 0.2788
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1160 - mae: 0.2561 - val_loss: 0.1244 - val_mae: 0.2652
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1057 - mae: 0.2433 - val_loss: 0.1166 - val_mae: 0.2549
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0990 - mae: 0.2344 - val_loss: 0.1122 - val_mae: 0.2486
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0944 - mae: 0.2279 - val_loss: 0.1079 - val_mae: 0.2445
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0910 - mae: 0.2229 - val_loss: 0.1044 - val_mae: 0.2390
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━━━━━

In [47]:
sgd_pred_scaled = rnn_sgd.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", sgd_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Prediction shape: (21713, 24)


In [48]:
sgd_pred = target_scaler.inverse_transform(
    sgd_pred_scaled.reshape(-1, 1)
).reshape(sgd_pred_scaled.shape)

sgd_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [49]:
sgd_mae = mean_absolute_error(sgd_actual.flatten(),sgd_pred.flatten())
sgd_rmse = np.sqrt(mean_squared_error( sgd_actual.flatten(),sgd_pred.flatten() ))
sgd_mape = np.mean(
    np.abs(
        (sgd_actual.flatten() - sgd_pred.flatten())
        / sgd_actual.flatten()
    )
) * 100

sgd_r2 = r2_score(sgd_actual.flatten(),sgd_pred.flatten())
sgd_bias = np.mean(sgd_pred.flatten() -sgd_actual.flatten())

print("=== RNN + SGD ===")
print("MAE :", sgd_mae)
print("RMSE:", sgd_rmse)
print("MAPE:", sgd_mape)
print("R²  :", sgd_r2)
print("Bias:", sgd_bias)

=== RNN + SGD ===
MAE : 1535.8984490213356
RMSE: 2110.63637839564
MAPE: 4.88878220366841
R²  : 0.8927331318248655
Bias: 136.84129949361895


**Learning Rate Scheduling**

In [50]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import RMSprop

rnn_lr = Sequential([SimpleRNN(64, input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_lr.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [51]:
history_lr = rnn_lr.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[lr_scheduler],
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.1252 - mae: 0.2568 - val_loss: 0.1033 - val_mae: 0.2331 - learning_rate: 0.0010
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0800 - mae: 0.2038 - val_loss: 0.0872 - val_mae: 0.2136 - learning_rate: 0.0010
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0739 - mae: 0.1937 - val_loss: 0.0945 - val_mae: 0.2200 - learning_rate: 0.0010
Epoch 4/15
1575/1588 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0705 - mae: 0.1881
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0704 - mae: 0.1881 - val_loss: 0.0903 - val_mae: 0.2158 - learning_rate: 0.0010
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0630 - mae: 0.1752 - val_loss: 0.0766 - val_mae: 0.1969 - learning_rate: 2.0000e-04
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0620 - mae: 0.1737 - val_loss: 0.0759 - val_mae: 0.1956 - learn

In [52]:
lr_pred_scaled = rnn_lr.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lr_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Prediction shape: (21713, 24)


In [53]:
lr_pred = target_scaler.inverse_transform(
    lr_pred_scaled.reshape(-1, 1)
).reshape(lr_pred_scaled.shape)

lr_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [54]:
lr_mae = mean_absolute_error(lr_actual.flatten(),lr_pred.flatten())
lr_rmse = np.sqrt(mean_squared_error(lr_actual.flatten(),lr_pred.flatten()))
lr_mape = np.mean(np.abs((lr_actual.flatten() - lr_pred.flatten())/ lr_actual.flatten())) * 100
lr_r2 = r2_score(lr_actual.flatten(),lr_pred.flatten())
lr_bias = np.mean(lr_pred.flatten() -lr_actual.flatten())

print("==== RNN + RMSPROP + LearningRate ====")
print("MAE :", lr_mae)
print("RMSE:", lr_rmse)
print("MAPE:", lr_mape)
print("R²  :", lr_r2)
print("Bias:", lr_bias)

==== RNN + RMSPROP + LearningRate ====
MAE : 1392.755252256083
RMSE: 1955.159110124279
MAPE: 4.417844875664829
R²  : 0.9079544124730492
Bias: 215.8540037600782


**layers, neurons**

In [55]:
rnn_deep = Sequential([
    SimpleRNN(
        64,
        return_sequences=True,
        input_shape=(48, 15)
    ),
    SimpleRNN(32,return_sequences=False ),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_deep.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_deep.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_12 (SimpleRNN)       │ (None, 48, 64)         │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_13 (SimpleRNN)       │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,896 (46.47 KB)

 Trainable params: 11,896 (46.47 KB)

 Non-trainable params: 0 (0.00 B)

In [56]:
history_deep = rnn_deep.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1252 - mae: 0.2538 - val_loss: 0.0957 - val_mae: 0.2280
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0768 - mae: 0.2000 - val_loss: 0.0887 - val_mae: 0.2150
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0717 - mae: 0.1918 - val_loss: 0.0886 - val_mae: 0.2136
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0688 - mae: 0.1869 - val_loss: 0.0848 - val_mae: 0.2096
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0668 - mae: 0.1836 - val_loss: 0.0825 - val_mae: 0.2022
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0659 - mae: 0.1822 - val_loss: 0.0776 - val_mae: 0.2003
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0644 - mae: 0.1798 - val_loss: 0.0790 - val_mae: 0.2031
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0631 - mae: 0.1781 - val_loss: 0.0776 - val_mae: 0.1992
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [57]:
deep_pred_scaled = rnn_deep.predict(
    X_test_seq,
    batch_size=64
)
print("Prediction shape:", deep_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Prediction shape: (21713, 24)


In [58]:
deep_pred = target_scaler.inverse_transform(
    deep_pred_scaled.reshape(-1, 1)
).reshape(deep_pred_scaled.shape)

deep_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [59]:
deep_mae = mean_absolute_error(deep_actual.flatten(), deep_pred.flatten())
deep_rmse = np.sqrt(mean_squared_error(deep_actual.flatten(),deep_pred.flatten()))
deep_mape = np.mean(np.abs((deep_actual.flatten() - deep_pred.flatten())/ deep_actual.flatten())) * 100
deep_r2 = r2_score(deep_actual.flatten(),deep_pred.flatten())
deep_bias = np.mean(deep_pred.flatten() - deep_actual.flatten())

print("========== DEEP RNN ==========")
print("MAE :", deep_mae)
print("RMSE:", deep_rmse)
print("MAPE:", deep_mape)
print("R²  :", deep_r2)
print("Bias:", deep_bias)

========== DEEP RNN ==========
MAE : 1418.3283535976911
RMSE: 1975.084423768742
MAPE: 4.502398797832909
R²  : 0.9060687524643493
Bias: 182.21977963410455


In [60]:
rnn_128 = Sequential([
    SimpleRNN(128, input_shape=(48, 15)),
    Dense( 64, activation="relu"),
    Dense(24)
])

rnn_128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_128.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_14 (SimpleRNN)       │ (None, 128)            │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,248 (110.34 KB)

 Trainable params: 28,248 (110.34 KB)

 Non-trainable params: 0 (0.00 B)

In [61]:
history_128 = rnn_128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.1081 - mae: 0.2361 - val_loss: 0.0914 - val_mae: 0.2196
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0743 - mae: 0.1956 - val_loss: 0.0813 - val_mae: 0.2052
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0690 - mae: 0.1867 - val_loss: 0.0808 - val_mae: 0.2055
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0662 - mae: 0.1821 - val_loss: 0.0797 - val_mae: 0.2047
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0637 - mae: 0.1779 - val_loss: 0.0789 - val_mae: 0.2029
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0625 - mae: 0.1763 - val_loss: 0.0763 - val_mae: 0.1967
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0616 - mae: 0.1746 - val_loss: 0.0771 - val_mae: 0.1996
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0605 - mae: 0.1732 - val_loss: 0.0784 - val_mae: 0.1981
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [62]:
rnn128_pred_scaled = rnn_128.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rnn128_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Prediction shape: (21713, 24)


In [63]:
rnn128_pred = target_scaler.inverse_transform(
    rnn128_pred_scaled.reshape(-1, 1)
).reshape(rnn128_pred_scaled.shape)

rnn128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [64]:
rnn128_mae = mean_absolute_error(rnn128_actual.flatten(),rnn128_pred.flatten())
rnn128_rmse = np.sqrt( mean_squared_error(rnn128_actual.flatten(), rnn128_pred.flatten()))
rnn128_mape = np.mean(
    np.abs(
        (rnn128_actual.flatten() - rnn128_pred.flatten())
        / rnn128_actual.flatten()
    )
) * 100

rnn128_r2 = r2_score(rnn128_actual.flatten(), rnn128_pred.flatten())
rnn128_bias = np.mean(rnn128_pred.flatten() - rnn128_actual.flatten())

print("=== RNN 128 NEURONS ===")
print("MAE :", rnn128_mae)
print("RMSE:", rnn128_rmse)
print("MAPE:", rnn128_mape)
print("R²  :", rnn128_r2)
print("Bias:", rnn128_bias)

=== RNN 128 NEURONS ===
MAE : 1422.0553110126457
RMSE: 2052.3782336090167
MAPE: 4.51038626512495
R²  : 0.8985730040929074
Bias: 349.6717504054131


**32 neurons**

In [65]:
rnn_32 = Sequential([SimpleRNN(32,input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_32.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_15 (SimpleRNN)       │ (None, 32)             │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,208 (20.34 KB)

 Trainable params: 5,208 (20.34 KB)

 Non-trainable params: 0 (0.00 B)

In [66]:
history_32 = rnn_32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.1580 - mae: 0.2866 - val_loss: 0.1040 - val_mae: 0.2387
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0883 - mae: 0.2173 - val_loss: 0.0962 - val_mae: 0.2242
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0794 - mae: 0.2039 - val_loss: 0.0890 - val_mae: 0.2147
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0751 - mae: 0.1968 - val_loss: 0.0851 - val_mae: 0.2087
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0725 - mae: 0.1926 - val_loss: 0.0840 - val_mae: 0.2078
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0710 - mae: 0.1898 - val_loss: 0.0839 - val_mae: 0.2077
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0698 - mae: 0.1879 - val_loss: 0.0822 - val_mae: 0.2057
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0686 - mae: 0.1858 - val_loss: 0.0846 - val_mae: 0.2093
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━━━━━

In [67]:
rnn32_pred_scaled = rnn_32.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rnn32_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 0s 903us/step
Prediction shape: (21713, 24)


In [68]:
rnn32_pred = target_scaler.inverse_transform(
    rnn32_pred_scaled.reshape(-1, 1)
).reshape(rnn32_pred_scaled.shape)

rnn32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [69]:
rnn32_mae = mean_absolute_error(rnn32_actual.flatten(), rnn32_pred.flatten())
rnn32_rmse = np.sqrt(mean_squared_error(rnn32_actual.flatten(), rnn32_pred.flatten()))
rnn32_mape = np.mean(np.abs((rnn32_actual.flatten() - rnn32_pred.flatten()) / rnn32_actual.flatten())) * 100
rnn32_r2 = r2_score(rnn32_actual.flatten(), rnn32_pred.flatten())
rnn32_bias = np.mean(rnn32_pred.flatten() - rnn32_actual.flatten())

print("=== RNN 32 NEURONS ===")
print("MAE :", rnn32_mae)
print("RMSE:", rnn32_rmse)
print("MAPE:", rnn32_mape)
print("R²  :", rnn32_r2)
print("Bias:", rnn32_bias)

=== RNN 32 NEURONS ===
MAE : 1447.132923811694
RMSE: 2001.175686149706
MAPE: 4.620242773169165
R²  : 0.9035706592429114
Bias: 313.53386593162196


**batch size(32)**

In [70]:
rnn_batch32 = Sequential([
    SimpleRNN(64, input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_batch32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch32 = rnn_batch32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=32,
    verbose=1
)

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.1139 - mae: 0.2425 - val_loss: 0.0921 - val_mae: 0.2212
Epoch 2/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0753 - mae: 0.1966 - val_loss: 0.0855 - val_mae: 0.2125
Epoch 3/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0701 - mae: 0.1881 - val_loss: 0.0811 - val_mae: 0.2023
Epoch 4/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0673 - mae: 0.1838 - val_loss: 0.0795 - val_mae: 0.2042
Epoch 5/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0657 - mae: 0.1808 - val_loss: 0.0780 - val_mae: 0.2017
Epoch 6/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.0644 - mae: 0.1786 - val_loss: 0.0803 - val_mae: 0.2046
Epoch 7/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0633 - mae: 0.1766 - val_loss: 0.0802 - val_mae: 0.2056
Epoch 8/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0631 - mae: 0.1764 - val_loss: 0.0745 - val_mae: 0.1928
Epoch 9/15
3175/3175 ━━━━━━━━━━━━━━━━━━

In [71]:
rnn_batch32_pred_scaled = rnn_batch32.predict(
    X_test_seq,
    batch_size=32
)

print("Prediction shape:", rnn_batch32_pred_scaled.shape)

679/679 ━━━━━━━━━━━━━━━━━━━━ 1s 907us/step
Prediction shape: (21713, 24)


In [72]:
rnnbatch32_pred = target_scaler.inverse_transform(
    rnn_batch32_pred_scaled.reshape(-1, 1)
).reshape(rnn_batch32_pred_scaled.shape)

rnnbatch32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [73]:
rnnb32_mae = mean_absolute_error(rnnbatch32_actual.flatten(),rnnbatch32_pred.flatten())
rnnb32_rmse = np.sqrt(mean_squared_error( rnnbatch32_actual.flatten(), rnnbatch32_pred.flatten()))
rnnb32_mape = np.mean(
    np.abs(
        (rnnbatch32_actual.flatten() - rnnbatch32_pred.flatten())
        / rnnbatch32_actual.flatten()
    )
) * 100
rnnb32_r2 = r2_score(rnnbatch32_actual.flatten(), rnnbatch32_pred.flatten())
rnnb32_bias = np.mean(rnnbatch32_pred.flatten() -rnnbatch32_actual.flatten())

print("=== RNN batch32 NEURONS ===")
print("MAE :", rnnb32_mae)
print("RMSE:", rnnb32_rmse)
print("MAPE:", rnnb32_mape)
print("R²  :", rnnb32_r2)
print("Bias:", rnnb32_bias)

=== RNN batch32 NEURONS ===
MAE : 1407.9610067104695
RMSE: 1988.6810486539075
MAPE: 4.46832158081662
R²  : 0.9047710419341246
Bias: 265.6179738817543


**batchsize128**

In [74]:
rnn_batch128 = Sequential([
    SimpleRNN(64, input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_batch128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch128 = rnn_batch128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=128,
    verbose=1
)

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 0.1521 - mae: 0.2799 - val_loss: 0.1022 - val_mae: 0.2384
Epoch 2/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.0836 - mae: 0.2105 - val_loss: 0.0875 - val_mae: 0.2136
Epoch 3/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.0742 - mae: 0.1955 - val_loss: 0.0861 - val_mae: 0.2129
Epoch 4/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.0706 - mae: 0.1894 - val_loss: 0.0819 - val_mae: 0.2044
Epoch 5/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.0678 - mae: 0.1847 - val_loss: 0.0795 - val_mae: 0.2030
Epoch 6/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.0662 - mae: 0.1820 - val_loss: 0.0802 - val_mae: 0.2036
Epoch 7/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.0647 - mae: 0.1795 - val_loss: 0.0764 - val_mae: 0.1973
Epoch 8/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.0637 - mae: 0.1779 - val_loss: 0.0776 - val_mae: 0.1998
Epoch 9/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - lo

In [75]:
rnn_batch128_pred_scaled = rnn_batch128.predict(
    X_test_seq,
    batch_size=128
)

print("Prediction shape:", rnn_batch128_pred_scaled.shape)

170/170 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Prediction shape: (21713, 24)


In [76]:
rnnbatch128_pred = target_scaler.inverse_transform(
    rnn_batch128_pred_scaled.reshape(-1, 1)
).reshape(rnn_batch128_pred_scaled.shape)

rnnbatch128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [77]:
rnnb128_mae = mean_absolute_error(rnnbatch128_actual.flatten(),rnnbatch128_pred.flatten())
rnnb128_rmse = np.sqrt(mean_squared_error( rnnbatch128_actual.flatten(), rnnbatch128_pred.flatten()))
rnnb128_mape = np.mean(
    np.abs(
        (rnnbatch128_actual.flatten() - rnnbatch128_pred.flatten())
        / rnnbatch128_actual.flatten()
    )
) * 100

rnnb128_r2 = r2_score(rnnbatch128_actual.flatten(), rnnbatch128_pred.flatten())
rnnb128_bias = np.mean(rnnbatch128_pred.flatten() -rnnbatch128_actual.flatten())

print("=== RNN batch128  ===")
print("MAE :", rnnb128_mae)
print("RMSE:", rnnb128_rmse)
print("MAPE:", rnnb128_mape)
print("R²  :", rnnb128_r2)
print("Bias:", rnnb128_bias)

=== RNN batch128  ===
MAE : 1374.1532677955986
RMSE: 1937.4937519839866
MAPE: 4.335888867529602
R²  : 0.9096102086861653
Bias: 86.10395423032861


**HyperParameter Tuning**

In [81]:
!pip install keras-tuner

zsh:1: command not found: pip


In [82]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [83]:
def build_rnn(hp):

    model = Sequential()
    model.add(SimpleRNN(units=hp.Choice(
                "rnn_units",
                values=[32, 64, 128]),
            input_shape=(48, 15)))

    model.add(
        Dense(
            units=hp.Choice(
                "dense_units",
                values=[32, 64, 128]
            ),
            activation="relu"
        )
    )

    model.add(Dense(24))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [84]:
tuner = kt.RandomSearch(
    build_rnn,
    objective="val_loss",
    max_trials=6,
    executions_per_trial=1,
    directory="hyperparameter_tuning",
    project_name="rnn_24_to_24"
)

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [85]:
tuner.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Trial 6 Complete [00h 01m 25s]
val_loss: 0.07439956068992615

Best val_loss So Far: 0.07306773960590363
Total elapsed time: 00h 11m 03s


In [86]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("BEST HYPERPARAMETERS")
print("RNN Units:",best_hp.get("rnn_units"))
print("Dense Units:",best_hp.get("dense_units"))
print("Learning Rate:",best_hp.get("learning_rate"))

BEST HYPERPARAMETERS
RNN Units: 128
Dense Units: 64
Learning Rate: 0.0005


In [87]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 16 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,248 (110.34 KB)

 Trainable params: 28,248 (110.34 KB)

 Non-trainable params: 0 (0.00 B)

In [88]:
history_best = best_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0566 - mae: 0.1670 - val_loss: 0.0745 - val_mae: 0.1910
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0557 - mae: 0.1656 - val_loss: 0.0751 - val_mae: 0.1941
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0544 - mae: 0.1642 - val_loss: 0.0827 - val_mae: 0.2050
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0536 - mae: 0.1632 - val_loss: 0.0784 - val_mae: 0.1961
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0524 - mae: 0.1617 - val_loss: 0.0764 - val_mae: 0.1934
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0525 - mae: 0.1623 - val_loss: 0.0805 - val_mae: 0.2032
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0521 - mae: 0.1618 - val_loss: 0.0771 - val_mae: 0.1953
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0503 - mae: 0.1594 - val_loss: 0.0737 - val_mae: 0.1918
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [89]:
best_pred_scaled = best_model.predict(
    X_test_seq,
    batch_size=64
)

best_pred = target_scaler.inverse_transform(
    best_pred_scaled.reshape(-1, 1)
).reshape(best_pred_scaled.shape)

best_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [90]:
best_mae = mean_absolute_error(best_actual.flatten(), best_pred.flatten())
best_rmse = np.sqrt( mean_squared_error( best_actual.flatten(), best_pred.flatten()))
best_mape = np.mean(
    np.abs(
        (best_actual.flatten() - best_pred.flatten())
        / best_actual.flatten()
    )
) * 100
best_r2 = r2_score(best_actual.flatten(),best_pred.flatten())
best_bias = np.mean(best_pred.flatten() -best_actual.flatten())

print("HYPERPARAMETER TUNED RNN")
print("MAE :", best_mae)
print("RMSE:", best_rmse)
print("MAPE:", best_mape)
print("R²  :", best_r2)
print("Bias:", best_bias)

HYPERPARAMETER TUNED RNN
MAE : 1463.8749714815085
RMSE: 2101.5526643314133
MAPE: 4.62568198378821
R²  : 0.8936544509182142
Bias: 167.92655028228697


**additional lags**

In [113]:
# Additional lag features

PJME_data["Lag_2"] = PJME_data["PJME_MW"].shift(2)
PJME_data["Lag_3"] = PJME_data["PJME_MW"].shift(3)
PJME_data["Lag_6"] = PJME_data["PJME_MW"].shift(6)
PJME_data["Lag_12"] = PJME_data["PJME_MW"].shift(12)
PJME_data["Lag_72"] = PJME_data["PJME_MW"].shift(72)
PJME_data["Lag_336"] = PJME_data["PJME_MW"].shift(336)

In [114]:
PJME_data = PJME_data.dropna().copy()

print("Missing values:")
print(PJME_data.isnull().sum().sum())

print("Shape:")
print(PJME_data.shape)

Missing values:
0
Shape:
(144552, 21)


In [115]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",

    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_6",
    "Lag_12",
    "Lag_24",
    "Lag_48",
    "Lag_72",
    "Lag_168",
    "Lag_336",

    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

print("Number of features:", len(features))

Number of features: 21


In [116]:
n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = PJME_data.iloc[:train_end]
val_data = PJME_data.iloc[train_end:val_end]
test_data = PJME_data.iloc[val_end:]

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (101186, 21)
Validation: (21683, 21)
Test: (21683, 21)


In [117]:
X_train = train_data[features]
y_train = train_data[["PJME_MW"]]

X_val = val_data[features]
y_val = val_data[["PJME_MW"]]

X_test = test_data[features]
y_test = test_data[["PJME_MW"]]

In [118]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(y_train)
y_val_scaled = target_scaler.transform(y_val)
y_test_scaled = target_scaler.transform(y_test)

print("Scaling completed.")

Scaling completed.


In [120]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=48,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=48,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=48,
    output_steps=24
)

print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101115, 48, 21)
y_train: (101115, 24)
X_val: (21612, 48, 21)
y_val: (21612, 24)
X_test: (21612, 48, 21)
y_test: (21612, 24)


In [121]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

rnn_lags = Sequential([SimpleRNN( 64, input_shape=(48, 21)),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_lags.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_lags.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │         5,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,224 (43.84 KB)

 Trainable params: 11,224 (43.84 KB)

 Non-trainable params: 0 (0.00 B)

In [122]:
history_lags = rnn_lags.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1259 - mae: 0.2532 - val_loss: 0.0938 - val_mae: 0.2221
Epoch 2/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0753 - mae: 0.1965 - val_loss: 0.0823 - val_mae: 0.2076
Epoch 3/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0700 - mae: 0.1880 - val_loss: 0.0803 - val_mae: 0.2037
Epoch 4/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0673 - mae: 0.1835 - val_loss: 0.0751 - val_mae: 0.1957
Epoch 5/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0656 - mae: 0.1808 - val_loss: 0.0768 - val_mae: 0.1967
Epoch 6/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0638 - mae: 0.1776 - val_loss: 0.0761 - val_mae: 0.1969
Epoch 7/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0633 - mae: 0.1769 - val_loss: 0.0755 - val_mae: 0.1982
Epoch 8/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0618 - mae: 0.1746 - val_loss: 0.0796 - val_mae: 0.1995
Epoch 9/15
1580/1580 ━━━━━━━━━━━━━━━━━━━

In [123]:
pred_scaled = rnn_lags.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Prediction shape: (21612, 24)


In [124]:
pred = target_scaler.inverse_transform(
    pred_scaled.reshape(-1, 1)
).reshape(pred_scaled.shape)

actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [125]:
mae = mean_absolute_error(actual.flatten(), pred.flatten())
rmse = np.sqrt(mean_squared_error(actual.flatten(), pred.flatten()))
mape = np.mean(np.abs((actual.flatten() - pred.flatten())/ actual.flatten())) * 100
r2 = r2_score(actual.flatten(), pred.flatten())
bias = np.mean(pred.flatten() - actual.flatten())

print("========== RNN + ADDITIONAL LAGS ==========")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

========== RNN + ADDITIONAL LAGS ==========
MAE : 1377.4734558824682
RMSE: 1930.5531219510005
MAPE: 4.382940742839425
R²  : 0.9101952997302649
Bias: 288.0007057432346


In [2]:
import pandas as pd

import pandas as pd
import os

rnn_48_24_results = [
    ["RNN Baseline", 48, 24, 1480.8368071696918, 2166.632234680274, 4.637196960894945, 0.8869659827518424, 244.60307681727969],
    ["RNN + EarlyStopping", 48, 24, 1391.5821917466278, 1947.6467640337682, 4.402527263388842, 0.9086603906880356, 115.37338664570908],
    ["RNN + Dropout", 48, 24, 1495.8575128838067, 2030.1111137647854, 4.816662229575421, 0.9007619139103995, 322.1394060022126],
    ["RNN + Batch Normalization", 48, 24, 1448.650549633674, 2017.3572154669084, 4.609178909509545, 0.90200489685922, 303.88435695490483],
    ["RNN + Adam", 48, 24, 1413.3843418354943, 1983.6639953866656, 4.456305317783067, 0.9052509239111448, 73.88685066067251],
    ["RNN + RMSprop", 48, 24, 1368.0419974047086, 1924.233401197823, 4.3164857429288785, 0.9108432435636404, 30.372446078949917],
    ["RNN + SGD", 48, 24, 1535.8984490213356, 2110.63637839564, 4.88878220366841, 0.8927331318248655, 136.84129949361895],
    ["RNN + RMSprop + Learning Rate", 48, 24, 1392.755252256083, 1955.159110124279, 4.417844875664829, 0.9079544124730492, 215.8540037600782],
    ["RNN + Additional Layers", 48, 24, 1418.3283535976911, 1975.084423768742, 4.502398797832909, 0.9060687524643493, 182.21977963410455],
    ["RNN 128 Neurons", 48, 24, 1422.0553110126457, 2052.3782336090167, 4.51038626512495, 0.8985730040929074, 349.6717504054131],
    ["RNN 32 Neurons", 48, 24, 1447.132923811694, 2001.175686149706, 4.620242773169165, 0.9035706592429114, 313.53386593162196],
    ["RNN Batch Size 32", 48, 24, 1407.9610067104695, 1988.6810486539075, 4.46832158081662, 0.9047710419341246, 265.6179738817543],
    ["RNN Batch Size 128", 48, 24, 1374.1532677955986, 1937.4937519839866, 4.335888867529602, 0.9096102086861653, 86.10395423032861],
    ["RNN Hyperparameter Tuned", 48, 24, 1463.8749714815085, 2101.5526643314133, 4.62568198378821, 0.8936544509182142, 167.92655028228697],
    ["RNN + Additional Lags", 48, 24, 1377.4734558824682, 1930.5531219510005, 4.382940742839425, 0.9101952997302649, 288.0007057432346]
]

rnn_48_24_df = pd.DataFrame(
    rnn_48_24_results,
    columns=[
        "Model",
        "Input Hours",
        "Output Hours",
        "MAE",
        "RMSE",
        "MAPE (%)",
        "R²",
        "Bias"
    ]
)

# Display with exactly 6 decimal places
display(
    rnn_48_24_df.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "MAPE (%)": "{:.6f}",
        "R²": "{:.6f}",
        "Bias": "{:.6f}"
    })
)

# Save inside existing comparison folder
os.makedirs("comparison", exist_ok=True)

rnn_48_24_df.to_csv(
    "comparison/RNN_48h_to_24h_results.csv",
    index=False,
    float_format="%.6f"
)

print("Saved: comparison/RNN_48h_to_24h_results.csv")



,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
0,RNN Baseline,48,24,1480.836807,2166.632235,4.637197,0.886966,244.603077
1,RNN + EarlyStopping,48,24,1391.582192,1947.646764,4.402527,0.908660,115.373387
2,RNN + Dropout,48,24,1495.857513,2030.111114,4.816662,0.900762,322.139406
3,RNN + Batch Normalization,48,24,1448.650550,2017.357215,4.609179,0.902005,303.884357
4,RNN + Adam,48,24,1413.384342,1983.663995,4.456305,0.905251,73.886851
5,RNN + RMSprop,48,24,1368.041997,1924.233401,4.316486,0.910843,30.372446
6,RNN + SGD,48,24,1535.898449,2110.636378,4.888782,0.892733,136.841299
7,RNN + RMSprop + Learning Rate,48,24,1392.755252,1955.159110,4.417845,0.907954,215.854004
8,RNN + Additional Layers,48,24,1418.328354,1975.084424,4.502399,0.906069,182.219780
9,RNN 128 Neurons,48,24,1422.055311,2052.378234,4.510386,0.898573,349.671750


Saved: comparison/RNN_48h_to_24h_results.csv


In [3]:
best_model = rnn_48_24_df.loc[rnn_48_24_df["R²"].idxmax()]
display(best_model.to_frame().T)

,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
5,RNN + RMSprop,48,24,1368.041997,1924.233401,4.316486,0.910843,30.372446
